# Síntesis de Voz: OpenAI TTS y ElevenLabs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/voz-audio/02-sintesis-voz.ipynb)

Síntesis de voz de calidad profesional: comparativa de proveedores, voces, streaming y pipeline Claude → TTS.

In [ ]:
!pip install openai anthropic elevenlabs -q

In [ ]:
import openai
import anthropic

openai_client = openai.OpenAI()
anthropic_client = anthropic.Anthropic()

## 1. OpenAI TTS — comparativa de voces

In [ ]:
VOCES_OPENAI = ['alloy', 'echo', 'fable', 'onyx', 'nova', 'shimmer']

def texto_a_voz_openai(texto: str, voz: str = 'nova', modelo: str = 'tts-1', ruta: str = 'audio.mp3') -> str:
    """Convierte texto a voz con OpenAI TTS."""
    respuesta = openai_client.audio.speech.create(
        model=modelo,
        voice=voz,
        input=texto,
    )
    respuesta.stream_to_file(ruta)
    return ruta

texto_demo = 'Hola, soy un asistente de inteligencia artificial. Estoy aquí para ayudarte.'

# Generar con todas las voces
for voz in VOCES_OPENAI:
    ruta = texto_a_voz_openai(texto_demo, voz=voz, ruta=f'/tmp/voz_{voz}.mp3')
    print(f'✅ {voz} → {ruta}')

## 2. Comparativa tts-1 vs tts-1-hd (calidad vs velocidad)

In [ ]:
import time

texto_largo = """
La inteligencia artificial está transformando todos los sectores de la economía.
Desde la medicina hasta el derecho, pasando por la educación y el entretenimiento,
los modelos de lenguaje están abriendo nuevas posibilidades que hace apenas
cinco años parecían imposibles. Este tutorial te enseña a integrar estas
capacidades en tus propios proyectos de forma práctica y eficiente.
""".strip()

for modelo in ['tts-1', 'tts-1-hd']:
    t0 = time.time()
    ruta = texto_a_voz_openai(texto_largo, voz='nova', modelo=modelo, ruta=f'/tmp/{modelo.replace("-","_")}.mp3')
    elapsed = time.time() - t0
    print(f'{modelo}: {elapsed:.2f}s → {ruta}')

print('\ntts-1: más rápido, suficiente para tiempo real')
print('tts-1-hd: mejor calidad, para producción y audiolibros')

## 3. Pipeline Claude → TTS

In [ ]:
import re

def limpiar_para_tts(texto: str) -> str:
    """Elimina markdown que no suena bien en audio."""
    texto = re.sub(r'\*\*(.*?)\*\*', r'\1', texto)
    texto = re.sub(r'\*(.*?)\*', r'\1', texto)
    texto = re.sub(r'`(.*?)`', r'\1', texto)
    texto = re.sub(r'#{1,6}\s', '', texto)
    texto = re.sub(r'^\s*[-•]\s', '', texto, flags=re.MULTILINE)
    return texto.strip()

def asistente_vocal(pregunta: str, historial: list | None = None, voz: str = 'nova') -> tuple:
    """Pipeline completo: pregunta → Claude → texto → TTS → audio."""
    mensajes = historial or []
    mensajes.append({'role': 'user', 'content': pregunta})
    
    # Claude genera la respuesta
    response = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        system='Eres un asistente vocal. Responde en 2-3 frases cortas y naturales, sin markdown ni listas.',
        messages=mensajes,
    )
    texto = limpiar_para_tts(response.content[0].text)
    
    # TTS convierte a audio
    audio = openai_client.audio.speech.create(model='tts-1', voice=voz, input=texto)
    ruta = '/tmp/respuesta.mp3'
    audio.stream_to_file(ruta)
    
    mensajes.append({'role': 'assistant', 'content': texto})
    return texto, ruta, mensajes

# Demo
preguntas = [
    '¿Qué es el prompt caching de Anthropic?',
    '¿Para qué sirve el Extended Thinking?',
]

historial = []
for pregunta in preguntas:
    texto, ruta, historial = asistente_vocal(pregunta, historial)
    print(f'Q: {pregunta}')
    print(f'A: {texto}')
    print(f'Audio: {ruta}\n')

## 4. Generador de audiolibro

In [ ]:
import os

def texto_a_audiolibro(texto: str, voz: str = 'fable', directorio: str = '/tmp/audiolibro') -> list:
    """Convierte texto largo en capítulos de audiolibro."""
    os.makedirs(directorio, exist_ok=True)
    
    # Dividir en fragmentos de ~800 chars (para TTS natural)
    fragmentos = [texto[i:i+800] for i in range(0, len(texto), 800)]
    archivos = []
    
    for i, fragmento in enumerate(fragmentos):
        if not fragmento.strip():
            continue
        ruta = f'{directorio}/parte_{i+1:03d}.mp3'
        audio = openai_client.audio.speech.create(model='tts-1-hd', voice=voz, input=fragmento.strip())
        audio.stream_to_file(ruta)
        archivos.append(ruta)
        print(f'  Parte {i+1}/{len(fragmentos)} → {ruta}')
    
    return archivos

# Demo con un extracto breve
texto_ejemplo = """
La historia de la inteligencia artificial comienza en 1950, cuando Alan Turing propuso
su famoso test para determinar si una máquina puede exhibir comportamiento inteligente
indistinguible del humano. Seis años después, en la conferencia de Dartmouth, se acuñó
oficialmente el término inteligencia artificial.

Durante las décadas siguientes, la IA atravesó periodos de gran optimismo seguidos de
los llamados inviernos de la IA, momentos en que la financiación y el interés decaían
ante la incapacidad de cumplir las expectativas prometidas.
""".strip()

archivos = texto_a_audiolibro(texto_ejemplo, voz='fable')
print(f'\nAudiolibro generado: {len(archivos)} parte(s)')